In [1]:
# Standard library imports
import os
import sys
import random
import warnings
import math

# Third-party numerical and data handling
import numpy as np
import pandas as pd
import h5py
import cv2
from PIL import Image

# Visualization
import matplotlib.pyplot as plt
from tqdm import tqdm

# Machine learning utilities
from sklearn.metrics import (
    average_precision_score,
    label_ranking_average_precision_score,
    roc_auc_score
)

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torch.amp import autocast, GradScaler

# PyTorch vision
from torchvision import models

# Albumentations
import albumentations as A
from albumentations.core.transforms_interface import ImageOnlyTransform
from albumentations.pytorch import ToTensorV2

# Custom modules (Kaggle inputs)
sys.path.append("/kaggle/input/asymmetric-loss-dataset")
sys.path.append("/kaggle/input/data-preprocessing")
sys.path.append("/kaggle/input/data-transformations")

from losses import AsymmetricLossOptimized
from preprocess import ecg_processing_pipeline, smart_pad_and_resize_ecg, ecg_processing_pipeline_no_perspective_distortion
from transformations import CornerCutout, GradientShadow, PaperFoldEffect, BottomBlur

### Helpers

In [2]:
def check_device():
    """
    Check available compute devices and return the best one.
    Priority: CUDA > MPS > CPU
    """
    if torch.cuda.is_available():
        device = torch.device("cuda")
        print("✓ CUDA available")
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
        print("✓ MPS (Apple Silicon GPU) available")
    else:
        device = torch.device("cpu")
        print("✗ Using CPU (no GPU acceleration available)")
    
    print(f"\nSelected device: {device}")
    return device

# Check and get device
device = check_device()

✓ CUDA available
  GPU: Tesla P100-PCIE-16GB
  Memory: 17.06 GB

Selected device: cuda


### Architecture

In [3]:
class Head(nn.Module):
    def __init__(self, in_features, hidden_layer, dropout_rate=0.3):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(in_features, hidden_layer),
            nn.BatchNorm1d(hidden_layer),  
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer, hidden_layer // 2), 
            nn.BatchNorm1d(hidden_layer // 2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_layer // 2, 1)
        )
    
    def forward(self, x):
        return self.layers(x)

class MultiHeadEfficientNet(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3):
        super().__init__()
        
        backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
        in_features = backbone.classifier[2].in_features
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"
        
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features), 
            nn.BatchNorm1d(in_features),
            nn.GELU(), 
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )
        
        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate) 
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]

### Image Preprocessing/DataLoader

In [4]:
def load_contours_from_hdf5(filepath='/kaggle/input/ecg-image-contours/contours.h5'):
    """
    Load all contours from HDF5 file back into dictionary format
    """
    contour_dict = {}
    
    with h5py.File(filepath, 'r') as f:
        for img_id in f.keys():
            grp = f[img_id]
            
            contour_dict[img_id] = {
                'contour': grp['contour'][:],  # Load the contour array
                'scale_x': grp.attrs['scale_x'],
                'scale_y': grp.attrs['scale_y'], 
                'half': grp.attrs['half']
            }
    
    return contour_dict

# Usage
try: 
    print(contours["train_000000"])
except Exception as e: 
    contours = load_contours_from_hdf5()

In [5]:
import time as time

def process_single_img(
                img,
                img_id,
                desired_aspect=0.5,
                target_width=512
                ):
    value = f"train_{str(img_id).zfill(6)}.png"
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    # output = ecg_processing_pipeline(input_image = img, 
    #                             contour_data = contours[value.split(".")[0]])
    output = ecg_processing_pipeline_no_perspective_distortion(input_image = img, 
                                contour_data = contours[value.split(".")[0]])
    resize_output = smart_pad_and_resize_ecg(output, target_size=(int(target_width*desired_aspect), 512), resize_strategy=cv2.INTER_AREA)
    return resize_output

def imread_clean(path):
    img = Image.open(path)
    img = img.convert('RGB')  # Strips metadata
    return np.array(img)

# def imread_clean(path):
#     with warnings.catch_warnings():
#         warnings.simplefilter('ignore')
#         img = cv2.imread(path)
#         if img is None:
#             raise ValueError(f"Failed to load image: {path}")
#         img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     return img
    
class ECGDataset(Dataset): 
    def __init__(self,
                image_paths, 
                labels_df, 
                transforms=None):
        
        self.image_paths = image_paths
        self.labels_dict = {idx: torch.tensor(row.values, dtype=torch.float32) 
                            for idx, row in labels_df.iterrows()}
        
        self.idx_to_image_id = {}
        for idx, path in enumerate(self.image_paths):
            filename = os.path.basename(path)
            image_id = int(filename.rsplit('_', 1)[-1].split('.')[0])
            self.idx_to_image_id[idx] = image_id
        
        self.transforms = transforms
        
    def __len__(self): 
        return len(self.image_paths)
        
    def __getitem__(self, idx): 
        time0 = time.time()
        image_path = self.image_paths[idx]
        index = self.idx_to_image_id[idx]
        label = self.labels_dict[index]
        time1 = time.time()
        image = imread_clean(image_path)
        if image is None:
            raise ValueError(f"Failed to load image: {image_path}")
        time2 = time.time()
        # convert image using our segmentation pipeline 
        image_conv = process_single_img(
            img=image, 
            img_id=index,
            desired_aspect=0.5, 
            target_width=512
        )
        image_conv = np.stack([image_conv, image_conv, image_conv], axis=-1)
        time3 = time.time()
        if self.transforms is not None: 
            image_tens = self.transforms(image=image_conv)["image"]
        else: 
            image_tens = image_conv
        time4 = time.time()
        #print(f"Collect: {t1-t0:.3f}s, Load: {t2-t1:.3f}s, Process: {t3-t2:.3f}s, Aug: {t4-t3:.3f}s")
        return image_tens, label

In [6]:
train_transforms = A.Compose([
    A.CLAHE(
        clip_limit=4,
        tile_grid_size=(8, 8),
        p=1.0
    ),
    #Data Augmentations
    A.Rotate(limit=2, p=0.3),  # small rotations, limit is +/- degrees
    A.Affine(translate_percent={'x': (-0.1, 0.1), 'y': (-0.05, 0.05)}, 
            rotate=0, scale=1.0, shear=0, p=0.3),  # small translations
    A.RandomShadow( # shadows
        shadow_roi=(0, 0, 1, 1),  # Can appear anywhere in image
        num_shadows_limit=(1,2),  # 1-2 shadow regions
        shadow_dimension=4,         # Controls shadow size/complexity
        shadow_intensity_range=(0.2, 0.4),
        p=0.3
    ),
    A.ElasticTransform(
        alpha=30, 
        sigma=15, 
        interpolation=cv2.INTER_AREA,
        p=0.3
    ),
    A.GaussianBlur(
        blur_limit=0, 
        sigma_limit=(0.1, 1.0),
        p=0.3
    ),
    A.Perspective(
        scale=[0.01, 0.03],
        keep_size=True,
        fit_output=True,
        interpolation=cv2.INTER_AREA,
        mask_interpolation=cv2.INTER_AREA,
        border_mode=cv2.BORDER_CONSTANT,
        fill=0,
        fill_mask=0,
        p=0.3
    ),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor and replicate grayscale to 3 channels
    ToTensorV2()
])

val_transforms = A.Compose([
    A.CLAHE(
        clip_limit=4,
        tile_grid_size=(8, 8),
        p=1.0
    ),
    # Normalization (ImageNet)
    A.Normalize(mean=[0.485, 0.456, 0.406], 
                std=[0.229, 0.224, 0.225]),
    # Convert to tensor
    ToTensorV2()
])

In [7]:
from sklearn.model_selection import train_test_split

labels_df = pd.read_csv("/kaggle/input/bhf-data-science-centre-ecg-challenge/train_final.csv", index_col=0, dtype=int)

with open("/kaggle/input/bhf-reference-files/broken_images_list.txt", "r") as file:
    broken_train_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/broken_test_images_list.txt", "r") as file:
    broken_test_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/valid_images_list.txt", "r") as file:
    valid_train_images = [line.strip() for line in file]
with open("/kaggle/input/bhf-reference-files/valid_test_images_list.txt", "r") as file:
    valid_test_images = [line.strip() for line in file]

ids = set([int(obj.split(".")[-2][-6:]) for obj in valid_train_images])

labels_df = labels_df.loc[labels_df.index.isin(ids)]

# split train set, using stratification 
X_train, X_test, y_train, y_test = train_test_split(valid_train_images, 
                                                    labels_df, 
                                                    test_size = 0.2,
                                                    random_state = 42, 
                                                    shuffle = True, 
                                                    stratify = labels_df[["CD", "MI", "AF", "STTC", "HYP"]])

In [8]:
num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

train_dataset = ECGDataset(
                    image_paths=X_train, 
                    labels_df=labels_df, 
                    transforms=train_transforms, 
                    )
val_dataset = ECGDataset(
                    image_paths=X_test, 
                    labels_df=labels_df, 
                    transforms=val_transforms, 
                    ) 

train_dataloader = DataLoader( 
                        train_dataset,
                        batch_size=32, 
                        shuffle=True, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)
val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

Using num_workers = 4


### Metrics for Evaluation

In [9]:
def batch_training_metrics(y_true, y_pred):
    """Compute sums that can be averaged later."""
    y_true = y_true.float()
    y_pred = y_pred.float()
    
    # Sum predicted probs per label
    sum_pred_prob_per_label = torch.sum(y_pred, dim=0)  # (L,)
    
    # Sum ground truth per label
    sum_true_per_label = torch.sum(y_true, dim=0)       # (L,)

    # Soft cardinality
    soft_cardinality_sum = torch.sum(torch.sum(y_pred, dim=1))  # scalar

    # Probability mass
    prob_mass_sum = torch.sum(y_pred)  # scalar
    
    # Calibration metrics - accumulate per label
    sum_pred_given_positive = torch.sum(y_pred * y_true, dim=0)  # (L,)
    sum_pred_given_negative = torch.sum(y_pred * (1 - y_true), dim=0)  # (L,)
    count_positive = torch.sum(y_true, dim=0)  # (L,)
    count_negative = torch.sum(1 - y_true, dim=0)  # (L,)

    return {
        "sum_pred_prob_per_label": sum_pred_prob_per_label,
        "sum_true_per_label": sum_true_per_label,
        "soft_cardinality_sum": soft_cardinality_sum,
        "prob_mass_sum": prob_mass_sum,
        "sum_pred_given_positive": sum_pred_given_positive,
        "sum_pred_given_negative": sum_pred_given_negative,
        "count_positive": count_positive,
        "count_negative": count_negative,
    }


def aggregate_training_epoch(batch_stats, total_samples):
    L = batch_stats[0]["sum_pred_prob_per_label"].shape[0]

    total_pred_prob = torch.zeros(L)
    total_true = torch.zeros(L)
    total_prob_mass = 0.0
    total_cardinality = 0.0
    total_pred_pos = torch.zeros(L)
    total_pred_neg = torch.zeros(L)
    total_count_pos = torch.zeros(L)
    total_count_neg = torch.zeros(L)

    for s in batch_stats:
        total_pred_prob += s["sum_pred_prob_per_label"].cpu()
        total_true += s["sum_true_per_label"].cpu()
        total_prob_mass += s["prob_mass_sum"].item()
        total_cardinality += s["soft_cardinality_sum"].item()
        total_pred_pos += s["sum_pred_given_positive"].cpu()
        total_pred_neg += s["sum_pred_given_negative"].cpu()
        total_count_pos += s["count_positive"].cpu()
        total_count_neg += s["count_negative"].cpu()
    
    mean_pred_when_positive = (total_pred_pos / (total_count_pos + 1e-8)).tolist()
    mean_pred_when_negative = (total_pred_neg / (total_count_neg + 1e-8)).tolist()

    return {
        "mean_pred_prob_per_label": (total_pred_prob / total_samples).tolist(),
        "mean_true_prob_per_label": (total_true / total_samples).tolist(),
        "mean_prob_mass": total_prob_mass / total_samples,
        "mean_cardinality": total_cardinality / total_samples,
        "mean_pred_when_positive": mean_pred_when_positive,
        "mean_pred_when_negative": mean_pred_when_negative,
        "calibration_gap": [(p - n) for p, n in zip(mean_pred_when_positive, mean_pred_when_negative)],
    }

def compute_ranking_metrics(all_y_true, all_y_pred):
    """Compute AP/AUROC/LRAP ranking metrics. Inputs are numpy arrays."""
    L = all_y_true.shape[1]

    per_label_ap = []
    per_label_auroc = []
    
    for j in range(L):
        # Average Precision
        ap = average_precision_score(all_y_true[:, j], all_y_pred[:, j])
        per_label_ap.append(float(ap))
        
        # AUROC
        try:
            auroc = roc_auc_score(all_y_true[:, j], all_y_pred[:, j])
            per_label_auroc.append(float(auroc))
        except ValueError:
            # Handle case where only one class is present in y_true
            per_label_auroc.append(float('nan'))

    # Micro-averaged metrics
    micro_ap = average_precision_score(all_y_true.reshape(-1), all_y_pred.reshape(-1))
    try:
        micro_auroc = roc_auc_score(all_y_true.reshape(-1), all_y_pred.reshape(-1))
    except ValueError:
        micro_auroc = float('nan')
    
    # Macro-averaged metrics
    macro_ap = sum(per_label_ap) / L
    valid_aurocs = [x for x in per_label_auroc if not np.isnan(x)]
    macro_auroc = sum(valid_aurocs) / len(valid_aurocs) if valid_aurocs else float('nan')
    
    # LRAP
    lrap = label_ranking_average_precision_score(all_y_true, all_y_pred)

    return {
        "per_label_ap": per_label_ap,
        "per_label_auroc": per_label_auroc,
        "macro_ap": float(macro_ap),
        "macro_auroc": float(macro_auroc),
        "micro_ap": float(micro_ap),
        "micro_auroc": float(micro_auroc),
        "lrap": float(lrap),
    }


### Initialise useful training functions

In [10]:
# load checkpoint 
checkpoint_path = "convnext_checkpoint.pth"
has_checkpoint = False
try: 
    checkpoint = torch.load(checkpoint_path, map_location=torch.device("cpu"), weights_only=False)
    has_checkpoint = True
    current_epoch = checkpoint["epoch"]
    print(f"Loading from checkpoint, last run epoch was {current_epoch}")
except Exception as e: 
    print("No checkpoint found, continuing as default")
    current_epoch = 0
    
num_epochs_decay = 80
num_epochs_frozen = 3
num_epochs_const = 10
num_warmup_epochs = 3
num_epochs_total = num_epochs_decay + num_epochs_frozen + num_epochs_const + num_warmup_epochs

No checkpoint found, continuing as default


In [11]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Multi-label focal loss with optional per-class alpha and gamma.
    Handles device placement automatically and ensures numerical stability.
    """

    def __init__(self, gamma=2.0, alpha=None, reduction="mean"):
        """
        Args:
            gamma (float or tensor): focusing parameter; scalar or per-class vector.
            alpha (float or tensor): class balance weights; scalar or per-class vector.
            reduction (str): "mean", "sum", or "none".
        """
        super().__init__()
        self.reduction = reduction

        # ---- Store gamma (scalar or vector) ----
        if torch.is_tensor(gamma):
            self.register_buffer("gamma", gamma.float())
            self.gamma_is_scalar = False
        else:
            self.gamma = float(gamma)
            self.gamma_is_scalar = True

        # ---- Store alpha (scalar or vector) ----
        if alpha is not None:
            if not torch.is_tensor(alpha):
                alpha = torch.tensor(alpha, dtype=torch.float32)
            self.register_buffer("alpha", alpha.float())
            self.alpha_is_set = True
        else:
            self.alpha_is_set = False

    def forward(self, logits, targets):
        """
        Args:
            logits: raw model outputs (batch, num_classes)
            targets: binary labels (batch, num_classes)
        """

        # ---- Numerically stable sigmoid + BCE ----
        # Instead of sigmoid(logits) then BCE, we use the built-in stable function.
        bce = F.binary_cross_entropy_with_logits(
            logits, targets, reduction="none"
        )

        # Stable sigmoid
        p = torch.sigmoid(logits)

        # p_t = p for y=1, else 1-p
        pt = p * targets + (1 - p) * (1 - targets)

        # ---- Make sure gamma and alpha match device and shape ----
        if self.gamma_is_scalar:
            gamma = self.gamma
        else:
            gamma = self.gamma.to(logits.device)  # (num_classes,)

        if self.alpha_is_set:
            alpha = self.alpha.to(logits.device)  # (num_classes,)
            alpha_t = alpha * targets + 1.0 * (1 - targets)
        else:
            alpha_t = 1.0

        # ---- Focal modulation ----
        # Add eps for numerical stability: (1 − pt) never becomes exactly 0
        eps = 1e-8
        focal_weight = (1 - pt + eps) ** gamma

        # ---- Apply alpha weighting ----
        focal_weight = focal_weight * alpha_t

        # ---- Combine focal term with BCE ----
        loss = focal_weight * bce

        # ---- Reduction ----
        if self.reduction == "mean":
            return loss.mean()
        elif self.reduction == "sum":
            return loss.sum()
        return loss

freq = labels_df.mean()
inv_freq = 1.0/freq
alpha = (inv_freq/inv_freq.max()).to_numpy()

criterion = FocalLoss(
    gamma = 2.0, 
    alpha = alpha,
    reduction="mean"
)

In [12]:
model = MultiHeadEfficientNet(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3
).to(device)

if has_checkpoint: 
    print("Loading model state dict from checkpoint")
    model.load_state_dict(checkpoint["model_state_dict"])

Downloading: "https://download.pytorch.org/models/convnext_base-6075fbad.pth" to /root/.cache/torch/hub/checkpoints/convnext_base-6075fbad.pth
100%|██████████| 338M/338M [00:01<00:00, 202MB/s]


In [13]:
# initially, we are going to freeze the weights of the backbone and just train new features
if current_epoch < num_epochs_frozen:
    for param in model.backbone.parameters():
        param.requires_grad = False 
    
opt = torch.optim.AdamW(
    [
        {"params": model.backbone.parameters(), "lr": 0.0},
        {"params": model.shared_feature_processor.parameters(), "lr": 1.0e-3},
        {"params": model.heads.parameters(), "lr": 1.0e-3}
    ],
    weight_decay = 1.0e-3
)

if has_checkpoint: 
    print("Loading up optimizer state dict")
    opt.load_state_dict(checkpoint["opt_state_dict"])

In [14]:
def get_lr(
        current_epoch, 
        frozen_epochs=num_epochs_frozen, 
        warmup_epochs=3, 
        decay_epochs=80,
        total_epochs=num_epochs_total, 
        min_lrs=[1.0e-6, 5.0e-6, 1.0e-5],
        max_lrs=[1.0e-4, 1.0e-3, 1.0e-3],
        ):
    
    out_lrs = {"backbone": None, "shared": None, "head": None}
    
    if current_epoch < frozen_epochs: 
        out_lrs["backbone"] = 0.0 
        out_lrs["shared"] = max_lrs[1]
        out_lrs["head"] = max_lrs[2]
        return out_lrs 
    
    if current_epoch < frozen_epochs + warmup_epochs: 
        inv_warmup_epochs = current_epoch - frozen_epochs
        out_lrs["backbone"] = max_lrs[0] * (inv_warmup_epochs / warmup_epochs)
        out_lrs["shared"] = max_lrs[1]
        out_lrs["head"] = max_lrs[2] 
        return out_lrs 
    
    if current_epoch < frozen_epochs + warmup_epochs + decay_epochs: 
        decay_epochs = frozen_epochs + warmup_epochs + decay_epochs - frozen_epochs - warmup_epochs
        decay_progress = (current_epoch - frozen_epochs - warmup_epochs) / decay_epochs
        decay_progress = min(max(decay_progress, 0), 1)  # clamp
        out_lrs["backbone"] = min_lrs[0] + (max_lrs[0] - min_lrs[0]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        out_lrs["shared"] = min_lrs[1] + (max_lrs[1] - min_lrs[1]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        out_lrs["head"] = min_lrs[2] + (max_lrs[2] - min_lrs[2]) * 0.5 * (1 + math.cos(math.pi * decay_progress))
        return out_lrs
    
    out_lrs["backbone"] = min_lrs[0]
    out_lrs["shared"] = min_lrs[1]
    out_lrs["head"] = min_lrs[2]
    
    return out_lrs

In [15]:
class EarlyStopping:
    """
    Early stops the training if validation loss doesn't improve after 'patience' epochs.
    Saves the best model automatically.
    """
    def __init__(self, patience=3, verbose=True, delta=0.0, best_loss=None, save_path="best_checkpoint.pth"):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.save_path = save_path
        
        self.best_loss = best_loss if best_loss is not None else float("inf")
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss, epoch, model, opt, extra_state=None):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            
            # save best model
            checkpoint = {
                "model_state_dict": model.state_dict(),
                "opt_state_dict": opt.state_dict(),
                "epoch": epoch,
            }
            if extra_state:
                checkpoint.update(extra_state)
            torch.save(checkpoint, self.save_path)
            
            if self.verbose:
                print(f"  ✓ Validation improved → saving new best model (loss={val_loss:.5f})")
        else:
            self.counter += 1
            if self.verbose:
                print(f"  ✗ No improvement ({self.counter}/{self.patience})")
            if self.counter >= self.patience:
                self.early_stop = True


In [16]:
accumulation_steps = 1 #  Effective batch size = batch_size * accumulation_steps
scaler = GradScaler('cuda') if device.type == "cuda" else None
train_losses = []
train_epoch_stats = []
val_losses = []
val_ranking_stats = []

early_stopper = EarlyStopping(
    patience=3,
    verbose=True,
    save_path="best_model.pth",
    best_loss=None
)
label_smoothing_eta = 0.03

def label_smoothing(labels, eta=label_smoothing_eta): 
    return labels * (1-eta) + 0.5 * eta

for epoch in range(current_epoch, num_epochs_total): 
        
    if epoch == num_epochs_frozen: 
        # unfreeze parameters
        for param in model.backbone.parameters(): 
            param.requires_grad = True 
            
    # get learning rates for current epoch
    lrs = get_lr(epoch)
    opt.param_groups[0]["lr"] = lrs["backbone"]
    opt.param_groups[1]["lr"] = lrs["shared"]
    opt.param_groups[2]["lr"] = lrs["head"]
    print("="*100)
    print(f"For epoch {epoch}, using learning rates {lrs}")
    
    model.train()
    opt.zero_grad()
    pbar = tqdm(total=len(train_dataloader),
                desc=f"Epoch {epoch} - Training", 
                unit="batch")
    running_train_loss = torch.tensor(0.0, device=device)
    total_samples = 0
    train_batch_stats = []
    for i, (inputs, labels) in enumerate(train_dataloader): 
        
        if (i + 1) % 100 == 0 or (i + 1) == len(train_dataloader):
            pbar.n = i + 1
            pbar.refresh()
        
        inputs = inputs.to(device)
        labels = labels.to(device)
        labels_smooth = label_smoothing(labels)
        
        if device.type == 'cuda':
            with autocast('cuda'): 
                outputs = model(inputs)
                loss = criterion(outputs, labels_smooth)
                loss = loss / accumulation_steps
            scaler.scale(loss).backward()
        else: 
            # doesn't support mixed precision training
            outputs = model(inputs)
            loss = criterion(outputs, labels_smooth)
            loss = loss / accumulation_steps
            loss.backward()
            
        with torch.no_grad():
            y_pred = torch.sigmoid(outputs.float())     # convert logits → probabilities
            stats = batch_training_metrics(labels, y_pred)
            train_batch_stats.append(stats)
            
        if (i + 1) % accumulation_steps == 0:
            if device.type == "cuda":
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
            else: 
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                opt.step()
                opt.zero_grad()
                
        # accumulate metrics 
        running_train_loss = running_train_loss + (loss.detach() * accumulation_steps * inputs.shape[0])
        total_samples += inputs.shape[0]
    # in the edge case where num_batches is not divisible by accumulation steps, need to do one further step 
    if (i + 1) % accumulation_steps != 0: 
        if device.type == "cuda":
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt)
            scaler.update()
            opt.zero_grad()
        else: 
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            opt.zero_grad()
        
    pbar.close()
        
    avg_train_loss = (running_train_loss / total_samples).item()
    train_losses.append({"epoch": epoch, "avg_train_loss": avg_train_loss})
    print(f"Epoch {epoch} Train Loss: ", avg_train_loss)
    train_stats_epoch = aggregate_training_epoch(train_batch_stats, total_samples)
    print(f"Epoch {epoch} TRAIN MONITOR:", train_stats_epoch)
    train_epoch_stats.append({"epoch": epoch, **train_stats_epoch})
    
    # VALIDATION LOOP   
    model.eval()
    pbar = tqdm(total=len(val_dataloader),
                desc=f"Epoch {epoch} - Validation", 
                unit="batch")
    running_val_loss = torch.tensor(0.0, device=device)
    total_samples = 0
    val_y_true_list = []
    val_y_pred_list = []
    with torch.no_grad(): 
        for i, (inputs, labels) in enumerate(val_dataloader): 
            if (i + 1) % 100 == 0 or (i + 1) == len(val_dataloader):
                pbar.n = i + 1
                pbar.refresh()

            inputs = inputs.to(device)
            labels = labels.to(device)
            
            if device.type == "cuda": 
                with autocast('cuda'): 
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            y_pred = torch.sigmoid(outputs)
            val_y_pred_list.append(y_pred.cpu())
            val_y_true_list.append(labels.cpu())
            
            running_val_loss = running_val_loss + (loss * inputs.shape[0])
            total_samples += inputs.shape[0]
        
    pbar.close()
    
    all_y_true = torch.cat(val_y_true_list).numpy()
    all_y_pred = torch.cat(val_y_pred_list).numpy()
    
    avg_val_loss = (running_val_loss / total_samples).item()
    print(f"Epoch {epoch} Val Loss: ", avg_val_loss)
    val_losses.append({"epoch": epoch, "avg_val_loss": avg_val_loss})
    
    ranking_metrics = compute_ranking_metrics(all_y_true, all_y_pred)
    print(f"Epoch {epoch} VALIDATION RANKING:", ranking_metrics)
    val_ranking_stats.append({"epoch": epoch, **ranking_metrics})

    early_stopper(
        val_loss=avg_val_loss,
        epoch=epoch,
        model=model,
        opt=opt,
        extra_state={"val_losses": val_losses, "train_losses": train_losses}
    )
        
    checkpoint = {
        "model_state_dict": model.state_dict(), 
        "opt_state_dict": opt.state_dict(), 
        "epoch": epoch+1, 
        "lrs": lrs, 
        "val_losses": val_losses, 
        "train_losses": train_losses}
    torch.save(checkpoint, "checkpoint.pth")

    train_stats_df = pd.DataFrame(train_epoch_stats)
    train_stats_df.to_csv("train_epoch_stats.csv", index=False)
    train_losses_df = pd.DataFrame(train_losses)
    train_losses_df.to_csv("train_losses.csv", index=False)
    val_stats_df = pd.DataFrame(val_ranking_stats)
    val_stats_df.to_csv("val_ranking_stats.csv", index=False)
    val_losses_df = pd.DataFrame(val_losses)
    val_losses_df.to_csv("val_losses.csv", index=False)
    
    if early_stopper.early_stop:
        print("Early stopping triggered. Training halting.")
        break

For epoch 0, using learning rates {'backbone': 0.0, 'shared': 0.001, 'head': 0.001}


Epoch 0 - Training: 100%|██████████| 376/376 [34:37<00:00,  5.53s/batch]


Epoch 0 Train Loss:  0.07676927000284195
Epoch 0 TRAIN MONITOR: {'mean_pred_prob_per_label': [0.2962808907032013, 0.27675217390060425, 0.2980756461620331, 0.29330432415008545, 0.27412477135658264], 'mean_true_prob_per_label': [0.24246209859848022, 0.12302181869745255, 0.2524571120738983, 0.20731301605701447, 0.06738297641277313], 'mean_prob_mass': 1.4385375334742703, 'mean_cardinality': 1.4385375337125699, 'mean_pred_when_positive': [0.30513519048690796, 0.2884282171726227, 0.301027774810791, 0.2945854067802429, 0.2780146300792694], 'mean_pred_when_negative': [0.29344698786735535, 0.2751140892505646, 0.2970784604549408, 0.29296910762786865, 0.27384352684020996], 'calibration_gap': [0.011688202619552612, 0.013314127922058105, 0.00394931435585022, 0.0016162991523742676, 0.004171103239059448]}


Epoch 0 - Validation: 100%|██████████| 94/94 [08:31<00:00,  5.44s/batch]


Epoch 0 Val Loss:  0.06741368770599365
Epoch 0 VALIDATION RANKING: {'per_label_ap': [0.41818871274828395, 0.312027089363092, 0.3182679374007634, 0.22439108886107692, 0.0910622419390517], 'per_label_auroc': [0.6651266224979946, 0.6548795299286062, 0.5568371019005888, 0.5017969593452716, 0.5426975663375502], 'macro_ap': 0.2727874140624536, 'macro_auroc': 0.5842675560020022, 'micro_ap': 0.24375652400614695, 'micro_auroc': 0.5896916505854692, 'lrap': 0.7634387260344948}
  ✓ Validation improved → saving new best model (loss=0.06741)
For epoch 1, using learning rates {'backbone': 0.0, 'shared': 0.001, 'head': 0.001}


...

Epoch 12 Val Loss:  0.035227950662374496
Epoch 12 VALIDATION RANKING: {'per_label_ap': [0.8062174060386531, 0.7242728260309128, 0.8313249363766302, 0.8348403378403445, 0.9006039625690946], 'per_label_auroc': [0.9286638493432691, 0.9155057479117739, 0.9180887783503734, 0.9186980904845817, 0.9912779299084644], 'macro_ap': 0.819451893771127, 'macro_auroc': 0.9344468791996924, 'micro_ap': 0.8139367172681508, 'micro_auroc': 0.9377827230764533, 'lrap': 0.9437550891997929}
  ✗ No improvement (2/3)
For epoch 13, using learning rates {'backbone': 9.814153420445554e-05, 'shared': 0.0009813214801356895, 'head': 0.0009814153420445552}


Epoch 13 - Training: 100%|██████████| 376/376 [39:06<00:00,  6.24s/batch]


Epoch 13 Train Loss:  0.03253394737839699
Epoch 13 TRAIN MONITOR: {'mean_pred_prob_per_label': [0.23588572442531586, 0.20271262526512146, 0.2500515282154083, 0.24609021842479706, 0.09380150586366653], 'mean_true_prob_per_label': [0.24246209859848022, 0.12302181869745255, 0.2524571120738983, 0.20731301605701447, 0.06738297641277313], 'mean_prob_mass': 1.0285416089473676, 'mean_cardinality': 1.0285416002097219, 'mean_pred_when_positive': [0.4589233994483948, 0.4609830379486084, 0.45893269777297974, 0.5067862272262573, 0.6526972055435181], 'mean_pred_when_negative': [0.16449899971485138, 0.16648267209529877, 0.1795092523097992, 0.17790985107421875, 0.05342048034071922], 'calibration_gap': [0.2944243997335434, 0.29450036585330963, 0.27942344546318054, 0.3288763761520386, 0.5992767252027988]}


Epoch 13 - Validation: 100%|██████████| 94/94 [08:33<00:00,  5.46s/batch]


Epoch 13 Val Loss:  0.03485054522752762
Epoch 13 VALIDATION RANKING: {'per_label_ap': [0.7930528021245211, 0.7111742785427436, 0.8161423746642106, 0.827724636148708, 0.8906864211892384], 'per_label_auroc': [0.9264475629910986, 0.9155863094357941, 0.9090213092843167, 0.916698989620558, 0.9915446942816003], 'macro_ap': 0.8077561025338843, 'macro_auroc': 0.9318597731226737, 'micro_ap': 0.7838988054473385, 'micro_auroc': 0.9334485385109462, 'lrap': 0.9352806462358442}
  ✗ No improvement (3/3)
Early stopping triggered. Training halting.
